# 05 · Final report: rebuilding the Part B numbers

This notebook rebuilds the numbers in Part B of the final report from the raw Weavve files. It writes each number
exactly as the report prints it and draws Figures 5 to 7. The last step looks for every rebuilt number in the report.

| Step | What it does |
|---|---|
| 1. Run the checks | Four scripts read the Weavve files and rebuild the numbers that Part B takes from them. |
| 2. Numbers by section | Each number appears as the report prints it, section by section. |
| 3. Figures | The notebook draws Figures 5 to 7 and saves them in the outputs folder. |
| 4. Report check | The notebook looks for each number in the report's Word file. |

This notebook does not rebuild Part A, which comes from the team's earlier notebooks. Table 11 and some Snooze Club
inputs come from the business team's own scripts, cited as [17] and [18] in the report.

## Setup

Put the Weavve files in `data/` first, as `data/README.md` describes. This notebook reads Files C, D, E, F, G, J, K,
L and 12, either as files or still inside the eLearn zip files.

Two more files are optional, and both stay in `data/` so that git never sees them. The ads monitor snapshots let the
notebook rebuild Section 10. The report's Word file lets it check every number word for word.

In [ ]:
import sys
sys.path.insert(0, "../src")          # our own helper code lives in the src folder

import json
import os
import runpy
from pathlib import Path

import pandas as pd
from IPython.display import display

from weavve import paths
from weavve.final import data, figures, report_numbers

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
CHECKS = Path(data.__file__).parent   # the check scripts sit next to data.py
FIGURES = paths.OUT / "final"         # Figures 5 to 7 are saved here
SNAPSHOT = "2026-10-31"               # the ads monitor snapshot that Section 10 of the report uses
print("Data folder:", paths.DATA)
print("Figures folder:", FIGURES)

## 1. Run the checks

Each check is a script in `src/weavve/final/`. It reads the raw files and rebuilds one group of numbers.

| Check | What it rebuilds |
|---|---|
| `check_recs.py` | It rebuilds the Section 6 findings, such as new website buyers and first-year value by starting channel. |
| `check_inputs.py` | It rebuilds newcomers, discount depth, the Snooze Club starting rates and what the ad account counts as a sale. |
| `check_team.py` | It checks the business team's Snooze Club figures one claim at a time. |
| `check_rec1.py` | It rebuilds our Recommendation 1 findings, such as how many new sign-ups buy within 90 days. |

The first check takes the longest, because it reads the order file for all four. The table under the code shows where
each file came from. It also shows whether each file matches the copy the report used.

In [ ]:
def run(check, name="res"):
    """Runs one check script and returns the results it builds."""
    return runpy.run_path(str(CHECKS / f"{check}.py"))[name]


R = {
    "recs": run("check_recs", "results"),
    "inputs": run("check_inputs"),
    "team": run("check_team"),
    "rec1": run("check_rec1"),
    "funnel": data.funnel_monthly(),
}
R = json.loads(json.dumps(R, default=str))   # text keys and plain values, as the checks first saved them
pd.DataFrame(data.SOURCES).T.rename(columns={"file": "Read from", "same_as_report": "Same copy as the report"})

## 2. Numbers by section

Each row is one number from Part B, written exactly as the report prints it. "Where" names the section or table.

Section 10 needs the three ads monitor snapshots from 31 October 2026. Without them, the notebook leaves Section 10
out.

In [ ]:
def snapshot(kind):
    """The ads monitor snapshot of this kind, or None if it is not in the data folder."""
    found = sorted(paths.DATA.rglob(f"ads_monitor_{kind}_{SNAPSHOT}.json"))
    return json.loads(found[0].read_text(encoding="utf-8")) if found else None


ads, log, plan = snapshot("api_data"), snapshot("api_log"), snapshot("plan")
q = report_numbers.quotes(R, ads, log, plan)
print(f"The checks rebuilt {len(q)} numbers.")
if not (ads and log and plan):
    print("The ads monitor snapshots are not in the data folder, so Section 10 is left out.")

In [ ]:
SECTIONS = {"6": "6. What We Found", "7": "7. Recommendation 1: Win First Website Orders",
            "8": "8. Recommendation 2: Snooze Club Worth Joining", "9": "9. Order of Actions and Shared Measures",
            "10": "10. Paid-Search Campaign", "11": "11. Assumptions and Limits"}
TABLE_SECTION = {10: "6", 11: "6", 12: "6", 13: "7", 14: "7", 15: "7", 16: "7", 17: "8", 18: "8", 19: "8", 20: "8",
                 21: "8", 22: "9", 25: "10", 26: "10"}
FIGURE_SECTION = {5: "6", 6: "7"}


def section(where):
    """The report section that holds a number, from its section number or its table."""
    first, second = where.split(" ")[:2]
    if first in ("Table", "Figure"):
        return (TABLE_SECTION if first == "Table" else FIGURE_SECTION)[int(second)]
    return first.split(".")[0]


rebuilt = pd.DataFrame({"Where": list(q), "As printed": list(q.values())})
rebuilt["Section"] = rebuilt["Where"].map(section)
for key, title in SECTIONS.items():
    part = rebuilt[rebuilt["Section"] == key]
    if len(part):
        print(title)
        display(part[["Where", "As printed"]].reset_index(drop=True))

## 3. Figures

These are Figures 5 to 7 of the report. The notebook also saves each one as a PNG in the figures folder from Setup.
Figure 7 needs the ads monitor snapshots.

In [ ]:
figures.figure5(R["recs"]["online_store"]["lens5"], FIGURES)
figures.figure6(R["rec1"]["new_signups_by_quarter"], FIGURES)
if ads and plan:
    figures.figure7(ads, plan["baseline"]["ctr"], FIGURES)

## 4. Check every number against the report

This step looks for each rebuilt number in the report's Word file. Put the file in `data/`, or set `WEAVVE_REPORT` to
its path. A number named after a table must appear in that table.

The repo is public, so it never holds the report's numbers. The report's own text is the answer key.

In [ ]:
found = [p for p in paths.DATA.rglob("*Final_Report*.docx") if not p.name.startswith("~$")]
if "WEAVVE_REPORT" in os.environ:
    report_file = Path(os.environ["WEAVVE_REPORT"])
else:
    report_file = max(found, key=os.path.getmtime, default=None)   # the most recently saved report

if report_file is None:
    print("The report's Word file is not in the data folder, so this check is skipped.")
else:
    missing = report_numbers.compare(q, report_numbers.report_text(report_file))
    print(f"{len(q) - len(missing)} of {len(q)} numbers appear in {report_file.name} exactly as rebuilt.")
    if missing:
        display(pd.DataFrame({"Where": list(missing), "Rebuilt": list(missing.values())}))